# tw-med-llm-qlora — Phase 4 A100 評估校準（CUDA 12.9 修正版）

這份 notebook **只執行 20 題 TMMLU+ validation 校準**，不讀取 test 題目，
也不會解鎖完整 28,758 次生成。它會用完全相同的 20 題依序測試：

1. 同尺寸原廠 instruct；
2. 台灣在地化 base；
3. 同一台灣 base + Phase 3 醫療 adapter。

A100 上以 vLLM 4-bit OpenAI-compatible server 執行，答案用 Twinkle Eval
`box` extractor 與 exact-match 檢查。完整題目和原始輸出只封存到私人 Drive ZIP；
可公開 manifest 只有 hash ID、答案標籤、摘要、耗時與成本估算。

此修正版固定安裝 vLLM 官方 CUDA 12.9 wheel，並在下載模型前驗證原生 CUDA
library 可匯入。舊版曾誤裝需要 `libcudart.so.13` 的 PyPI wheel。

> 研究用途，非醫療建議。請先刪除舊 runtime，再從新 A100 runtime 按
>「全部執行」，不要從中段開始，也不要沿用已載入舊 CUDA 套件的 session。


## 1. 安裝鎖定依賴

本節使用 vLLM 官方 release 的 `cu129` wheel，並交由 uv 選擇相符的 PyTorch
CUDA 12.9 套件。請務必從乾淨 runtime 執行；不需要手改任何程式碼。


In [ ]:
import json
import subprocess
import sys

NOTEBOOK_BUILD = "phase4-calibration-policy-v4"
already_loaded = sorted(
    name for name in ("torch", "vllm") if name in sys.modules
)
if already_loaded:
    raise RuntimeError(
        "This notebook must start in a fresh Colab runtime before changing the "
        f"CUDA stack; already loaded: {already_loaded}. Use Runtime > "
        "Disconnect and delete runtime, reconnect to A100, then Run all."
    )

uv_requirement = "uv==0.11.31"
runtime_requirements = json.loads(
    "[\"vllm @ https://github.com/vllm-project/vllm/releases/download/v0.25.1/vllm-0.25.1%2Bcu129-cp38-abi3-manylinux_2_28_x86_64.whl#sha256=9e206f370c934a2d4b6b1f05d3d09708d344e05d80260189ef19f60755709431\", \"twinkle-eval==2.8.0\", \"bitsandbytes==0.49.2\"]"
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", uv_requirement],
    check=True,
)
install_command = [
    "uv",
    "pip",
    "install",
    "--system",
    "--upgrade",
    "--torch-backend",
    "cu129",
    *runtime_requirements,
]
print(json.dumps({
    "notebook_build": NOTEBOOK_BUILD,
    "torch_backend": "cu129",
    "vllm_wheel": "https://github.com/vllm-project/vllm/releases/download/v0.25.1/vllm-0.25.1%2Bcu129-cp38-abi3-manylinux_2_28_x86_64.whl",
}, indent=2))
subprocess.run(install_command, check=True)


In [ ]:
import importlib.metadata
import importlib.util
import json
import subprocess
import sys

REQUIRED_EVAL_PACKAGES = {
    "vllm": ("vllm", "0.25.1+cu129"),
    "twinkle-eval": ("twinkle_eval", "2.8.0"),
    "bitsandbytes": ("bitsandbytes", "0.49.2"),
}
dependency_audit = {}
dependency_errors = []
for package_name, (module_name, expected_version) in REQUIRED_EVAL_PACKAGES.items():
    module_available = importlib.util.find_spec(module_name) is not None
    try:
        installed_version = importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    dependency_audit[package_name] = {
        "module": module_name,
        "module_available": module_available,
        "expected_version": expected_version,
        "installed_version": installed_version,
    }
    if not module_available or installed_version != expected_version:
        dependency_errors.append(package_name)
print(json.dumps(dependency_audit, ensure_ascii=False, indent=2))
if dependency_errors:
    raise RuntimeError(
        "Phase 4 dependencies are missing or mismatched: "
        f"{dependency_errors}. Delete the runtime and run this notebook "
        "from the top; do not repair the environment in place."
    )

native_probe_code = r'''
import importlib.metadata
import json
import torch
import vllm

payload = {
    "vllm_version": importlib.metadata.version("vllm"),
    "torch_version": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cuda_available": torch.cuda.is_available(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
print(json.dumps(payload, sort_keys=True))
'''
native_probe = subprocess.run(
    [sys.executable, "-c", native_probe_code],
    capture_output=True,
    text=True,
    timeout=180,
)
if native_probe.returncode != 0:
    print(native_probe.stdout)
    print(native_probe.stderr)
    raise RuntimeError(
        "vLLM native CUDA preflight failed before any model download. "
        "Delete the runtime, reconnect to A100, and run this corrected "
        "notebook from the top."
    )
native_lines = [line for line in native_probe.stdout.splitlines() if line.strip()]
native_audit = json.loads(native_lines[-1])
if native_audit["torch_cuda"] != "12.9" or not native_audit["cuda_available"]:
    raise RuntimeError(f"Unexpected CUDA runtime after installation: {native_audit}")
dependency_audit["native_cuda_preflight"] = native_audit
print(json.dumps({
    "notebook_build": "phase4-calibration-cu129-v2",
    "vllm_native_import": True,
    **native_audit,
}, ensure_ascii=False, indent=2))


## 2. 固定設定、Secrets、Drive 與 A100 硬體閘門

不需要修改 code。只需在 Colab Secrets 開啟 `HF_TOKEN` 的 notebook 存取權，並使用
A100 GPU。完整評估硬閘門在這份 notebook 中固定關閉。


In [ ]:
import concurrent.futures
import hashlib
import json
import os
import platform
import shutil
import signal
import subprocess
import sys
import time
import urllib.error
import urllib.request
from datetime import UTC, datetime
from pathlib import Path

import torch
from google.colab import drive, userdata
from huggingface_hub import snapshot_download

PROJECT_CONFIG = json.loads(r'''{"data":{"medqa":{"config":"med_qa_tw_source","dataset_id":"bigbio/med_qa","expected_test_rows":1413,"expected_train_rows":11248,"expected_validation_rows":1409,"revision":"e04abdc0672c54547fa1dbe36cfefc000e4f2657","sample_size":5,"source_sha256":{"test":"799f6c7881d50b1101f168f31129aea699fec67401e2b8dc9464f1f84dc40c04","train":"b2cb4b82a7c7ffc6f6f99465197f4dfc94b04464cf2ab517a67418641c617646","validation":"06c2334173f17ed0665088e920bbf0fb54bb64ef7432885747668235ed54701a"}},"tmmluplus":{"dataset_id":"ikala/tmmluplus","revision":"81d53e38340c9ade988f7fed8996da6554b504f3"}},"evaluation":{"bootstrap_iterations":10000,"calibration_examples":20,"control_subjects":["chinese_language_and_literature","geography_of_taiwan","logic_reasoning","computer_science","general_principles_of_law"],"forgetting_margin_percentage_points":2.0,"full_approval":{"approval_phrase":"確認解鎖 Phase 4 正式評估","approved":true,"approved_at":"2026-07-22","approved_requests":28758,"calibration_manifest_sha256":"d9c0f23e72808f0a8fc4edc1a7889719637f9390346725d5f72f10c4d3e2cdf2","calibration_run_id":"20260722T061028Z","calibration_validation_sha256":"6c39aff8cd4e82b80d24a4ce7f7959e7b7fd5ba9546f2432ef3b7c96212b2d85","compute_units_per_hour":5.3,"parallel_workers":4,"projected_compute_units":15.791642498050743,"projected_compute_units_with_20pct_buffer":18.94997099766089,"projected_hours":2.979555188311461,"required_approval_code":"PHASE4_FULL_28758_APPROVED_20260722","shard_size":250},"full_shuffle_seed":3407,"generation":{"max_tokens":256,"minimum_calibration_parse_rate":0.8,"temperature":0.0,"token_limit_hits_count_as_incorrect":true,"token_limit_hits_fail_calibration":false,"top_p":1.0},"medical_subjects":["basic_medical_science","clinical_psychology","dentistry","occupational_therapy_for_psychological_disorders","optometry","pharmacology","pharmacy","traditional_chinese_medicine_clinical_medicine"],"phase3_adapter":{"archive_bytes":113079186,"archive_sha256":"2c537dfd3049319286c678a3ca3aa72e3f20baa7e0f44bde93ff7ee4dc47e43e","base_model_id":"taide/Gemma-3-TAIDE-12b-Chat-2602","drive_archive":"/content/drive/MyDrive/tw-med-llm-qlora/phase3/141226999eacb67ffd9a/full/runs/20260722T014216Z-phase3-full.zip","selected_checkpoint":700},"stability_examples_per_subject":100,"stability_seeds":[3407,3408,3409],"twinkle_eval":{"extractor":"box","repository":"https://github.com/ai-twinkle/Eval","revision":"470bbec130fa95c9e2e06c6a4b06db4a51390a06","scorer":"exact_match","version":"2.8.0"},"vllm":{"expected_torch_cuda":"12.9","gpu_memory_utilization":0.85,"language_model_only":true,"max_lora_rank":16,"max_model_length":2048,"quantization":"bitsandbytes","torch_backend":"cu129","uv_version":"0.11.31","version":"0.25.1","wheel_sha256":"9e206f370c934a2d4b6b1f05d3d09708d344e05d80260189ef19f60755709431","wheel_url":"https://github.com/vllm-project/vllm/releases/download/v0.25.1/vllm-0.25.1%2Bcu129-cp38-abi3-manylinux_2_28_x86_64.whl","wheel_variant":"cu129"},"workload":{"expected_medqa_full_requests":4239,"expected_tmmlu_full_requests":16719,"expected_tmmlu_stability_requests":7800,"expected_total_requests":28758,"full_model_count":3,"medqa_test_rows":1413,"stability_model_count":2,"tmmlu_test_rows":5573}},"export":{"gguf":{"approval_phrase":"確認解鎖 GGUF Q4_K_M A100 匯出，上限 6.36 CU；產物只存 Google Drive，不上傳外部，完成後執行 Windows Ollama 驗收","approved_at":"2026-07-23","approved_compute_units_with_20pct_buffer":6.36,"compute_units_per_hour":5.3,"drive_root":"/content/drive/MyDrive/tw-med-llm-qlora/phase5/gguf-q4-k-m/runs","enabled":true,"expected_gguf_gib_lower":7.0,"expected_gguf_gib_upper":9.0,"external_upload_allowed":false,"maximum_memory_usage":0.5,"minimum_drive_disk_gib":20.0,"minimum_local_disk_gib":100.0,"minimum_vram_gib":38.0,"ollama_model_name":"tw-med-taide-12b-q4-k-m","projected_compute_units_lower":2.65,"projected_compute_units_upper":5.3,"projected_hours_lower":0.5,"projected_hours_upper":1.0,"quantization_method":"q4_k_m","required_approval_code":"GGUF_Q4_K_M_A100_APPROVED_20260723","required_gpu_name_contains":"A100","run_environment":"colab_linux"}},"hardware_profiles":[{"allow_tf32":true,"batch_size":8,"gradient_accumulation_steps":2,"max_sequence_length":2048,"min_compute_capability":[8,0],"min_vram_gib":70.0,"model_profile":"primary","name":"primary_80g","requires_bf16":true},{"allow_tf32":true,"batch_size":4,"gradient_accumulation_steps":4,"max_sequence_length":2048,"min_compute_capability":[8,0],"min_vram_gib":38.0,"model_profile":"primary","name":"primary_40g","requires_bf16":true},{"allow_tf32":true,"batch_size":1,"gradient_accumulation_steps":16,"max_sequence_length":2048,"min_compute_capability":[8,0],"min_vram_gib":22.0,"model_profile":"primary","name":"primary_24g","requires_bf16":true},{"allow_tf32":false,"batch_size":2,"gradient_accumulation_steps":8,"max_sequence_length":2048,"min_compute_capability":[7,5],"min_vram_gib":14.0,"model_profile":"fallback","name":"fallback_16g","requires_bf16":false}],"inference":{"phase3_adapter":{"archive_sha256":"2c537dfd3049319286c678a3ca3aa72e3f20baa7e0f44bde93ff7ee4dc47e43e","base_model_id":"taide/Gemma-3-TAIDE-12b-Chat-2602","base_model_revision":"4de0b93b99f8b61b59c40d019fd593bdd1c42249","selected_checkpoint":700},"windows_4090":{"attention_implementation":"sdpa","double_quantization":true,"load_in_4bit":true,"max_new_tokens":64,"minimum_compute_capability":[8,9],"minimum_vram_gib":22.0,"ollama_model_name":"tw-med-taide-12b","quantization_type":"nf4","required_gpu_name":"RTX 4090","requires_bf16":true}},"models":{"fallback":{"baseline_id":"google/gemma-3-4b-it","baseline_revision":"093f9f388b31de276ce2de164bdc2081324b9767","batch_size":2,"gradient_accumulation_steps":8,"max_sequence_length":2048,"model_id":"twinkle-ai/gemma-3-4B-T1-it","requires_bf16":false,"revision":"63b2bb819a7885b476c2ff98a418de8400661f9d"},"primary":{"baseline_id":"google/gemma-3-12b-it","baseline_revision":"96b6f1eccf38110c56df3a15bffe176da04bfd80","batch_size":1,"gradient_accumulation_steps":16,"max_sequence_length":2048,"model_id":"taide/Gemma-3-TAIDE-12b-Chat-2602","requires_bf16":true,"revision":"4de0b93b99f8b61b59c40d019fd593bdd1c42249"}},"project":{"effective_batch_size":16,"seed":3407},"publication":{"adapter_repo_id":"steven0226/tw-med-llm-qlora-adapter","enabled":true,"github_repository_url":"https://github.com/kuotunyu/tw-med-llm-qlora","requires_explicit_repo_id":true,"requires_explicit_visibility_confirmation":true,"visibility":"private"},"training":{"eval_steps":100,"learning_rate":5e-05,"load_in_4bit":true,"lora_alpha":16,"lora_dropout":0.0,"lora_rank":16,"lr_scheduler_type":"cosine","num_train_epochs":1,"optimizer":"adamw_8bit","save_steps":100,"save_total_limit":2,"smoke_examples":100,"smoke_steps":10,"warmup_ratio":0.03}}''')
RUN_MODE = "calibration"
ALLOW_FULL_EVALUATION = False
FULL_EVALUATION_APPROVAL = None
REQUIRED_FULL_EVALUATION_APPROVAL = "PHASE4_28758_REQUESTS"
COMPUTE_UNITS_PER_HOUR = 5.3
PRICE_PER_COMPUTE_UNIT = None
CURRENCY_LABEL = None

if RUN_MODE != "calibration":
    raise RuntimeError("This reviewed notebook permits calibration mode only")
if ALLOW_FULL_EVALUATION or FULL_EVALUATION_APPROVAL is not None:
    raise RuntimeError(
        "Full Phase 4 evaluation remains locked until calibration review"
    )

if not torch.cuda.is_available():
    raise RuntimeError("Phase 4 calibration requires an A100 GPU runtime")
gpu_name = torch.cuda.get_device_name(0)
gpu_properties = torch.cuda.get_device_properties(0)
gpu_vram_gib = gpu_properties.total_memory / 1024**3
if "A100" not in gpu_name.upper() or gpu_vram_gib < 38:
    raise RuntimeError(
        f"Expected A100 >=38 GiB; detected {gpu_name} ({gpu_vram_gib:.2f} GiB)"
    )
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The reviewed Phase 4 profile requires BF16 support")

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError(
        "Colab Secret HF_TOKEN is missing or not enabled for this notebook"
    )
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
drive.mount("/content/drive")

RUN_ID = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
RUN_ROOT = Path("/content/tw-med-phase4-calibration") / RUN_ID
PRIVATE_ROOT = RUN_ROOT / "private"
PUBLIC_ROOT = RUN_ROOT / "public"
LOG_ROOT = PRIVATE_ROOT / "server-logs"
for directory in (PRIVATE_ROOT, PUBLIC_ROOT, LOG_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

hardware_audit = {
    "gpu_name": gpu_name,
    "gpu_vram_gib": gpu_vram_gib,
    "compute_capability": list(torch.cuda.get_device_capability(0)),
    "bf16_supported": torch.cuda.is_bf16_supported(),
    "torch_version": torch.__version__,
    "cuda_version": torch.version.cuda,
    "platform": platform.platform(),
}
print(json.dumps(hardware_audit, ensure_ascii=False, indent=2))


## 3. 載入 repo 測試過的評估 helper


In [ ]:
# ruff: noqa: E402, E501, I001
EMBEDDED_HELPER_FILES = json.loads("{\"tw_med_qlora/__init__.py\": \"\", \"tw_med_qlora/types.py\": \"\\\"\\\"\\\"Stable, dependency-free project data types.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom collections.abc import Mapping\\nfrom dataclasses import dataclass\\n\\nCHOICE_KEYS = (\\\"A\\\", \\\"B\\\", \\\"C\\\", \\\"D\\\")\\nSPLITS = frozenset({\\\"train\\\", \\\"validation\\\", \\\"test\\\"})\\n\\n\\ndef stable_example_id(\\n    *,\\n    source: str,\\n    revision: str,\\n    split: str,\\n    question: str,\\n    choices: Mapping[str, str],\\n) -> str:\\n    \\\"\\\"\\\"Create a deterministic ID without exposing the question text.\\\"\\\"\\\"\\n\\n    payload = {\\n        \\\"source\\\": source,\\n        \\\"revision\\\": revision,\\n        \\\"split\\\": split,\\n        \\\"question\\\": question,\\n        \\\"choices\\\": {key: choices[key] for key in CHOICE_KEYS},\\n    }\\n    canonical = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(\\\",\\\", \\\":\\\"))\\n    return hashlib.sha256(canonical.encode(\\\"utf-8\\\")).hexdigest()[:20]\\n\\n\\n@dataclass(frozen=True)\\nclass MCQExample:\\n    \\\"\\\"\\\"Canonical four-choice example shared by preparation and evaluation.\\\"\\\"\\\"\\n\\n    id: str\\n    source: str\\n    revision: str\\n    split: str\\n    question: str\\n    choices: Mapping[str, str]\\n    answer: str\\n\\n    def __post_init__(self) -> None:\\n        if self.split not in SPLITS:\\n            raise ValueError(f\\\"Unsupported split: {self.split}\\\")\\n        if not self.question.strip():\\n            raise ValueError(\\\"question must not be empty\\\")\\n        if tuple(sorted(self.choices)) != CHOICE_KEYS:\\n            raise ValueError(\\\"choices must contain exactly A, B, C, and D\\\")\\n        if any(not self.choices[key].strip() for key in CHOICE_KEYS):\\n            raise ValueError(\\\"choice text must not be empty\\\")\\n        if self.answer not in CHOICE_KEYS:\\n            raise ValueError(\\\"answer must be one of A, B, C, or D\\\")\\n\", \"tw_med_qlora/medqa.py\": \"\\\"\\\"\\\"Strict MedQA parsing and content-safe validation helpers.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom collections.abc import Iterable, Iterator, Mapping\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path\\nfrom typing import Any, TextIO\\n\\nfrom tw_med_qlora.types import CHOICE_KEYS, MCQExample, stable_example_id\\n\\nMEDQA_FIELDS = frozenset({\\\"meta_info\\\", \\\"question\\\", \\\"answer_idx\\\", \\\"answer\\\", \\\"options\\\"})\\n\\n\\nclass MedQAValidationError(ValueError):\\n    \\\"\\\"\\\"Raised when a source row cannot be converted without ambiguity.\\\"\\\"\\\"\\n\\n\\n@dataclass(frozen=True)\\nclass DuplicateRemoval:\\n    \\\"\\\"\\\"Content-safe provenance for a lower-priority duplicate.\\\"\\\"\\\"\\n\\n    example_id: str\\n    removed_split: str\\n    winner_split: str\\n    question_key_sha256: str\\n\\n\\ndef _utf8_text(value: object, *, field: str) -> str:\\n    if not isinstance(value, str):\\n        raise MedQAValidationError(f\\\"{field} must be a string\\\")\\n    if not value.strip():\\n        raise MedQAValidationError(f\\\"{field} must not be empty\\\")\\n    try:\\n        if value.encode(\\\"utf-8\\\").decode(\\\"utf-8\\\") != value:\\n            raise MedQAValidationError(f\\\"{field} failed UTF-8 round trip\\\")\\n    except UnicodeError as error:\\n        raise MedQAValidationError(f\\\"{field} is not valid UTF-8 text\\\") from error\\n    return value\\n\\n\\ndef medqa_row_to_example(\\n    row: Mapping[str, Any],\\n    *,\\n    split: str,\\n    source: str,\\n    revision: str,\\n) -> MCQExample:\\n    \\\"\\\"\\\"Convert one source-format row and reject malformed or ambiguous answers.\\\"\\\"\\\"\\n\\n    missing = MEDQA_FIELDS.difference(row)\\n    if missing:\\n        raise MedQAValidationError(f\\\"missing fields: {', '.join(sorted(missing))}\\\")\\n\\n    question = _utf8_text(row[\\\"question\\\"], field=\\\"question\\\")\\n    answer = _utf8_text(row[\\\"answer_idx\\\"], field=\\\"answer_idx\\\").strip().upper()\\n    answer_text = _utf8_text(row[\\\"answer\\\"], field=\\\"answer\\\")\\n\\n    raw_options = row[\\\"options\\\"]\\n    if not isinstance(raw_options, list):\\n        raise MedQAValidationError(\\\"options must be a list\\\")\\n\\n    choices: dict[str, str] = {}\\n    for index, option in enumerate(raw_options):\\n        if not isinstance(option, Mapping):\\n            raise MedQAValidationError(f\\\"options[{index}] must be an object\\\")\\n        key = _utf8_text(option.get(\\\"key\\\"), field=f\\\"options[{index}].key\\\").strip().upper()\\n        value = _utf8_text(option.get(\\\"value\\\"), field=f\\\"options[{index}].value\\\")\\n        if key in choices:\\n            raise MedQAValidationError(f\\\"duplicate option key: {key}\\\")\\n        choices[key] = value\\n\\n    if tuple(sorted(choices)) != CHOICE_KEYS:\\n        raise MedQAValidationError(\\\"options must contain exactly A, B, C, and D\\\")\\n    if answer not in choices:\\n        raise MedQAValidationError(\\\"answer_idx must be one of A, B, C, or D\\\")\\n    if answer_text != choices[answer]:\\n        raise MedQAValidationError(\\\"answer text does not match the selected option\\\")\\n\\n    example_id = stable_example_id(\\n        source=source,\\n        revision=revision,\\n        split=split,\\n        question=question,\\n        choices=choices,\\n    )\\n    return MCQExample(\\n        id=example_id,\\n        source=source,\\n        revision=revision,\\n        split=split,\\n        question=question,\\n        choices=choices,\\n        answer=answer,\\n    )\\n\\n\\ndef iter_parquet_rows(path: Path, *, limit: int | None = None) -> Iterator[dict[str, Any]]:\\n    \\\"\\\"\\\"Stream source rows from Parquet without loading a full split into memory.\\\"\\\"\\\"\\n\\n    import pyarrow.parquet as parquet\\n\\n    if limit is not None and limit <= 0:\\n        raise ValueError(\\\"limit must be positive\\\")\\n\\n    emitted = 0\\n    parquet_file = parquet.ParquetFile(path)\\n    for batch in parquet_file.iter_batches(batch_size=min(limit or 1024, 1024)):\\n        for row in batch.to_pylist():\\n            yield row\\n            emitted += 1\\n            if limit is not None and emitted >= limit:\\n                return\\n\\n\\ndef content_fingerprint(examples: Iterable[MCQExample]) -> str:\\n    \\\"\\\"\\\"Hash canonical private content without returning or persisting that content.\\\"\\\"\\\"\\n\\n    digest = hashlib.sha256()\\n    for example in examples:\\n        payload = {\\n            \\\"id\\\": example.id,\\n            \\\"source\\\": example.source,\\n            \\\"revision\\\": example.revision,\\n            \\\"split\\\": example.split,\\n            \\\"question\\\": example.question,\\n            \\\"choices\\\": dict(example.choices),\\n            \\\"answer\\\": example.answer,\\n        }\\n        digest.update(\\n            json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(\\\",\\\", \\\":\\\")).encode(\\n                \\\"utf-8\\\"\\n            )\\n        )\\n        digest.update(b\\\"\\\\n\\\")\\n    return digest.hexdigest()\\n\\n\\ndef normalized_question(question: str) -> str:\\n    \\\"\\\"\\\"Conservatively normalize whitespace and case without folding medical symbols.\\\"\\\"\\\"\\n\\n    return \\\" \\\".join(question.split()).casefold()\\n\\n\\ndef has_ambiguous_choice_text(example: MCQExample) -> bool:\\n    \\\"\\\"\\\"Detect two option keys whose normalized visible text is identical.\\\"\\\"\\\"\\n\\n    normalized_values = [normalized_question(example.choices[key]) for key in CHOICE_KEYS]\\n    return len(set(normalized_values)) != len(normalized_values)\\n\\n\\ndef deduplicate_splits(\\n    examples_by_split: Mapping[str, list[MCQExample]],\\n) -> tuple[dict[str, list[MCQExample]], list[DuplicateRemoval]]:\\n    \\\"\\\"\\\"Apply test > validation > train priority while preserving test unchanged.\\\"\\\"\\\"\\n\\n    expected_splits = {\\\"train\\\", \\\"validation\\\", \\\"test\\\"}\\n    if set(examples_by_split) != expected_splits:\\n        raise ValueError(f\\\"expected splits: {sorted(expected_splits)}\\\")\\n\\n    cleaned: dict[str, list[MCQExample]] = {\\n        \\\"test\\\": list(examples_by_split[\\\"test\\\"]),\\n        \\\"validation\\\": [],\\n        \\\"train\\\": [],\\n    }\\n    removals: list[DuplicateRemoval] = []\\n    seen: dict[str, str] = {}\\n\\n    for example in cleaned[\\\"test\\\"]:\\n        seen.setdefault(normalized_question(example.question), \\\"test\\\")\\n\\n    for split in (\\\"validation\\\", \\\"train\\\"):\\n        for example in examples_by_split[split]:\\n            key = normalized_question(example.question)\\n            winner_split = seen.get(key)\\n            if winner_split is not None:\\n                removals.append(\\n                    DuplicateRemoval(\\n                        example_id=example.id,\\n                        removed_split=split,\\n                        winner_split=winner_split,\\n                        question_key_sha256=hashlib.sha256(key.encode(\\\"utf-8\\\")).hexdigest(),\\n                    )\\n                )\\n                continue\\n            cleaned[split].append(example)\\n            seen[key] = split\\n\\n    return cleaned, removals\\n\\n\\ndef count_within_split_duplicate_rows(examples: Iterable[MCQExample]) -> int:\\n    \\\"\\\"\\\"Count repeat rows after the first occurrence without removing them.\\\"\\\"\\\"\\n\\n    seen: set[str] = set()\\n    duplicate_rows = 0\\n    for example in examples:\\n        key = normalized_question(example.question)\\n        if key in seen:\\n            duplicate_rows += 1\\n        else:\\n            seen.add(key)\\n    return duplicate_rows\\n\\n\\ndef duplicate_removals_fingerprint(removals: Iterable[DuplicateRemoval]) -> str:\\n    \\\"\\\"\\\"Fingerprint ordered removal provenance without exposing source content.\\\"\\\"\\\"\\n\\n    digest = hashlib.sha256()\\n    for removal in removals:\\n        digest.update(\\n            json.dumps(asdict(removal), sort_keys=True, separators=(\\\",\\\", \\\":\\\")).encode(\\\"utf-8\\\")\\n        )\\n        digest.update(b\\\"\\\\n\\\")\\n    return digest.hexdigest()\\n\\n\\ndef assert_split_isolation(examples_by_split: Mapping[str, list[MCQExample]]) -> None:\\n    \\\"\\\"\\\"Assert normalized questions and stable IDs do not cross split boundaries.\\\"\\\"\\\"\\n\\n    question_keys: dict[str, set[str]] = {}\\n    ids: dict[str, set[str]] = {}\\n    for split, examples in examples_by_split.items():\\n        question_keys[split] = {normalized_question(item.question) for item in examples}\\n        ids[split] = {item.id for item in examples}\\n\\n    splits = (\\\"train\\\", \\\"validation\\\", \\\"test\\\")\\n    for index, left in enumerate(splits):\\n        for right in splits[index + 1 :]:\\n            if question_keys[left].intersection(question_keys[right]):\\n                raise ValueError(f\\\"normalized question overlap remains: {left}/{right}\\\")\\n            if ids[left].intersection(ids[right]):\\n                raise ValueError(f\\\"stable ID overlap remains: {left}/{right}\\\")\\n\\n\\ndef example_record(example: MCQExample) -> dict[str, Any]:\\n    \\\"\\\"\\\"Return the fixed serialized MCQ schema.\\\"\\\"\\\"\\n\\n    record = asdict(example)\\n    record[\\\"choices\\\"] = {key: example.choices[key] for key in CHOICE_KEYS}\\n    return record\\n\\n\\ndef _write_jsonl_rows(output_file: TextIO, examples: Iterable[MCQExample]) -> None:\\n    for example in examples:\\n        output_file.write(\\n            json.dumps(\\n                example_record(example),\\n                ensure_ascii=False,\\n                sort_keys=True,\\n                separators=(\\\",\\\", \\\":\\\"),\\n            )\\n        )\\n        output_file.write(\\\"\\\\n\\\")\\n\\n\\ndef write_jsonl_atomic(path: Path, examples: Iterable[MCQExample]) -> None:\\n    \\\"\\\"\\\"Write deterministic UTF-8 JSONL and replace only after a complete write.\\\"\\\"\\\"\\n\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    temporary = path.with_suffix(f\\\"{path.suffix}.tmp\\\")\\n    with temporary.open(\\\"w\\\", encoding=\\\"utf-8\\\", newline=\\\"\\\\n\\\") as output_file:\\n        _write_jsonl_rows(output_file, examples)\\n    temporary.replace(path)\\n\\n\\ndef file_sha256(path: Path) -> str:\\n    \\\"\\\"\\\"Hash a file in bounded chunks.\\\"\\\"\\\"\\n\\n    digest = hashlib.sha256()\\n    with path.open(\\\"rb\\\") as source_file:\\n        for chunk in iter(lambda: source_file.read(1024 * 1024), b\\\"\\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\", \"tw_med_qlora/evaluation.py\": \"\\\"\\\"\\\"Dependency-free helpers shared by smoke tests and formal evaluation.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport math\\nimport random\\nimport re\\nfrom collections import defaultdict\\nfrom collections.abc import Iterable, Sequence\\nfrom dataclasses import asdict, dataclass\\nfrom typing import Any\\n\\n_STRICT_DIRECT_ANSWER = re.compile(r\\\"\\\\A\\\\s*([A-D])\\\\s*[\u3002.]?\\\\s*\\\\Z\\\")\\n_BOXED_ANSWER = re.compile(r\\\"\\\\\\\\boxed\\\\s*\\\\{\\\\s*([A-D])\\\\s*\\\\}\\\")\\n_BOX_MARKER = re.compile(r\\\"\\\\\\\\boxed\\\\b\\\")\\n_SHA256 = re.compile(r\\\"\\\\A[0-9a-f]{64}\\\\Z\\\")\\n\\n\\ndef parse_mcq_answer(text: str) -> str | None:\\n    \\\"\\\"\\\"Return one unambiguous A-D answer under the evaluation contract.\\n\\n    A response is valid when it is either a standalone uppercase choice or contains\\n    exactly one simple LaTeX box whose entire payload is an uppercase choice. The\\n    latter supports benchmark models that reason before emitting ``\\\\\\\\boxed{A}``.\\n    Missing, malformed, nested, or multiple boxes are rejected.\\n    \\\"\\\"\\\"\\n\\n    direct = _STRICT_DIRECT_ANSWER.fullmatch(text)\\n    if direct is not None:\\n        return direct.group(1)\\n\\n    box_markers = _BOX_MARKER.findall(text)\\n    valid_boxes = _BOXED_ANSWER.findall(text)\\n    if len(box_markers) == 1 and len(valid_boxes) == 1:\\n        return valid_boxes[0]\\n    return None\\n\\n\\n@dataclass(frozen=True)\\nclass PredictionRecord:\\n    \\\"\\\"\\\"Content-safe per-question result used for public statistical summaries.\\\"\\\"\\\"\\n\\n    example_id: str\\n    model: str\\n    source: str\\n    subject: str\\n    gold: str\\n    prediction: str | None\\n    raw_output_sha256: str\\n    latency_seconds: float\\n    prompt_tokens: int | None = None\\n    completion_tokens: int | None = None\\n\\n    def __post_init__(self) -> None:\\n        if not self.example_id:\\n            raise ValueError(\\\"example_id must not be empty\\\")\\n        if not self.model:\\n            raise ValueError(\\\"model must not be empty\\\")\\n        if not self.source:\\n            raise ValueError(\\\"source must not be empty\\\")\\n        if self.gold not in {\\\"A\\\", \\\"B\\\", \\\"C\\\", \\\"D\\\"}:\\n            raise ValueError(\\\"gold must be one of A, B, C, or D\\\")\\n        if self.prediction is not None and self.prediction not in {\\\"A\\\", \\\"B\\\", \\\"C\\\", \\\"D\\\"}:\\n            raise ValueError(\\\"prediction must be None or one of A, B, C, or D\\\")\\n        if not _SHA256.fullmatch(self.raw_output_sha256):\\n            raise ValueError(\\\"raw_output_sha256 must be a lowercase SHA-256 digest\\\")\\n        if not math.isfinite(self.latency_seconds) or self.latency_seconds < 0:\\n            raise ValueError(\\\"latency_seconds must be finite and non-negative\\\")\\n        for name, count in (\\n            (\\\"prompt_tokens\\\", self.prompt_tokens),\\n            (\\\"completion_tokens\\\", self.completion_tokens),\\n        ):\\n            if count is not None and count < 0:\\n                raise ValueError(f\\\"{name} must be non-negative\\\")\\n\\n    @property\\n    def parsed(self) -> bool:\\n        return self.prediction is not None\\n\\n    @property\\n    def correct(self) -> bool:\\n        return self.prediction == self.gold\\n\\n    def as_public_dict(self) -> dict[str, Any]:\\n        \\\"\\\"\\\"Serialize without question text or raw model output.\\\"\\\"\\\"\\n\\n        record = asdict(self)\\n        record[\\\"parsed\\\"] = self.parsed\\n        record[\\\"correct\\\"] = self.correct\\n        return record\\n\\n\\ndef prediction_record(\\n    *,\\n    example_id: str,\\n    model: str,\\n    source: str,\\n    subject: str,\\n    gold: str,\\n    raw_output: str,\\n    latency_seconds: float,\\n    prompt_tokens: int | None = None,\\n    completion_tokens: int | None = None,\\n) -> PredictionRecord:\\n    \\\"\\\"\\\"Parse one strict A-D response while retaining only its digest publicly.\\\"\\\"\\\"\\n\\n    return PredictionRecord(\\n        example_id=example_id,\\n        model=model,\\n        source=source,\\n        subject=subject,\\n        gold=gold,\\n        prediction=parse_mcq_answer(raw_output),\\n        raw_output_sha256=hashlib.sha256(raw_output.encode(\\\"utf-8\\\")).hexdigest(),\\n        latency_seconds=latency_seconds,\\n        prompt_tokens=prompt_tokens,\\n        completion_tokens=completion_tokens,\\n    )\\n\\n\\ndef _unique_by_id(records: Iterable[PredictionRecord]) -> dict[str, PredictionRecord]:\\n    indexed: dict[str, PredictionRecord] = {}\\n    for record in records:\\n        if record.example_id in indexed:\\n            raise ValueError(f\\\"duplicate prediction ID: {record.example_id}\\\")\\n        indexed[record.example_id] = record\\n    if not indexed:\\n        raise ValueError(\\\"at least one prediction record is required\\\")\\n    return indexed\\n\\n\\ndef accuracy_summary(records: Iterable[PredictionRecord]) -> dict[str, int | float]:\\n    \\\"\\\"\\\"Report accuracy with parse failures counted as incorrect.\\\"\\\"\\\"\\n\\n    indexed = _unique_by_id(records)\\n    values = list(indexed.values())\\n    parsed = sum(record.parsed for record in values)\\n    correct = sum(record.correct for record in values)\\n    total = len(values)\\n    return {\\n        \\\"total\\\": total,\\n        \\\"parsed\\\": parsed,\\n        \\\"parse_failures\\\": total - parsed,\\n        \\\"correct\\\": correct,\\n        \\\"accuracy\\\": correct / total,\\n        \\\"parse_rate\\\": parsed / total,\\n    }\\n\\n\\ndef subject_accuracy(\\n    records: Iterable[PredictionRecord],\\n) -> dict[str, dict[str, int | float]]:\\n    \\\"\\\"\\\"Return stable alphabetical subject summaries.\\\"\\\"\\\"\\n\\n    grouped: dict[str, list[PredictionRecord]] = defaultdict(list)\\n    for record in records:\\n        if not record.subject:\\n            raise ValueError(\\\"subject_accuracy requires non-empty subjects\\\")\\n        grouped[record.subject].append(record)\\n    if not grouped:\\n        raise ValueError(\\\"at least one prediction record is required\\\")\\n    return {subject: accuracy_summary(grouped[subject]) for subject in sorted(grouped)}\\n\\n\\ndef _aligned_pairs(\\n    base_records: Iterable[PredictionRecord],\\n    adapter_records: Iterable[PredictionRecord],\\n) -> list[tuple[PredictionRecord, PredictionRecord]]:\\n    base = _unique_by_id(base_records)\\n    adapter = _unique_by_id(adapter_records)\\n    if set(base) != set(adapter):\\n        missing_adapter = sorted(set(base).difference(adapter))\\n        missing_base = sorted(set(adapter).difference(base))\\n        raise ValueError(\\n            \\\"paired prediction IDs differ: \\\"\\n            f\\\"missing_adapter={missing_adapter[:3]}, missing_base={missing_base[:3]}\\\"\\n        )\\n\\n    pairs: list[tuple[PredictionRecord, PredictionRecord]] = []\\n    for example_id in sorted(base):\\n        left = base[example_id]\\n        right = adapter[example_id]\\n        if (left.gold, left.source, left.subject) != (\\n            right.gold,\\n            right.source,\\n            right.subject,\\n        ):\\n            raise ValueError(f\\\"paired prediction metadata differs: {example_id}\\\")\\n        pairs.append((left, right))\\n    return pairs\\n\\n\\ndef _percentile(sorted_values: Sequence[float], quantile: float) -> float:\\n    if not sorted_values:\\n        raise ValueError(\\\"percentile requires values\\\")\\n    if not 0 <= quantile <= 1:\\n        raise ValueError(\\\"quantile must be between zero and one\\\")\\n    position = (len(sorted_values) - 1) * quantile\\n    lower = math.floor(position)\\n    upper = math.ceil(position)\\n    if lower == upper:\\n        return float(sorted_values[lower])\\n    weight = position - lower\\n    return float(sorted_values[lower] * (1 - weight) + sorted_values[upper] * weight)\\n\\n\\ndef paired_bootstrap_accuracy_difference(\\n    base_records: Iterable[PredictionRecord],\\n    adapter_records: Iterable[PredictionRecord],\\n    *,\\n    iterations: int = 10_000,\\n    seed: int = 3407,\\n    stratify_by_subject: bool = False,\\n) -> dict[str, int | float | bool]:\\n    \\\"\\\"\\\"Bootstrap adapter-minus-base accuracy using paired question outcomes.\\n\\n    When ``stratify_by_subject`` is true, questions are resampled within each subject and\\n    the replicate statistic is the unweighted mean of subject-level differences.\\n    \\\"\\\"\\\"\\n\\n    if iterations <= 0:\\n        raise ValueError(\\\"iterations must be positive\\\")\\n    pairs = _aligned_pairs(base_records, adapter_records)\\n    grouped: dict[str, list[int]] = defaultdict(list)\\n    for base, adapter in pairs:\\n        subject = base.subject if stratify_by_subject else \\\"__all__\\\"\\n        if stratify_by_subject and not subject:\\n            raise ValueError(\\\"stratified bootstrap requires non-empty subjects\\\")\\n        grouped[subject].append(int(adapter.correct) - int(base.correct))\\n\\n    observed = sum(sum(values) / len(values) for values in grouped.values()) / len(grouped)\\n    rng = random.Random(seed)\\n    replicates: list[float] = []\\n    for _ in range(iterations):\\n        subject_differences = []\\n        for values in grouped.values():\\n            sampled_sum = sum(values[rng.randrange(len(values))] for _ in values)\\n            subject_differences.append(sampled_sum / len(values))\\n        replicates.append(sum(subject_differences) / len(subject_differences))\\n    replicates.sort()\\n\\n    return {\\n        \\\"iterations\\\": iterations,\\n        \\\"seed\\\": seed,\\n        \\\"stratified_by_subject\\\": stratify_by_subject,\\n        \\\"observed_difference_percentage_points\\\": observed * 100,\\n        \\\"ci_lower_percentage_points\\\": _percentile(replicates, 0.025) * 100,\\n        \\\"ci_upper_percentage_points\\\": _percentile(replicates, 0.975) * 100,\\n    }\\n\\n\\ndef mcnemar_exact_test(\\n    base_records: Iterable[PredictionRecord],\\n    adapter_records: Iterable[PredictionRecord],\\n) -> dict[str, int | float]:\\n    \\\"\\\"\\\"Return the two-sided exact McNemar test for paired correctness outcomes.\\\"\\\"\\\"\\n\\n    pairs = _aligned_pairs(base_records, adapter_records)\\n    improved = sum((not base.correct) and adapter.correct for base, adapter in pairs)\\n    regressed = sum(base.correct and (not adapter.correct) for base, adapter in pairs)\\n    discordant = improved + regressed\\n    if discordant == 0:\\n        p_value = 1.0\\n    else:\\n        tail = min(improved, regressed)\\n        log_probabilities = [\\n            math.lgamma(discordant + 1)\\n            - math.lgamma(k + 1)\\n            - math.lgamma(discordant - k + 1)\\n            - discordant * math.log(2)\\n            for k in range(tail + 1)\\n        ]\\n        maximum = max(log_probabilities)\\n        cdf = math.exp(maximum) * sum(\\n            math.exp(value - maximum) for value in log_probabilities\\n        )\\n        p_value = min(1.0, 2 * cdf)\\n    return {\\n        \\\"base_wrong_adapter_correct\\\": improved,\\n        \\\"base_correct_adapter_wrong\\\": regressed,\\n        \\\"discordant_pairs\\\": discordant,\\n        \\\"two_sided_exact_p_value\\\": p_value,\\n    }\\n\\n\\ndef forgetting_noninferiority(\\n    base_records: Iterable[PredictionRecord],\\n    adapter_records: Iterable[PredictionRecord],\\n    *,\\n    margin_percentage_points: float = 2.0,\\n    iterations: int = 10_000,\\n    seed: int = 3407,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Apply the predeclared subject-macro catastrophic-forgetting rule.\\\"\\\"\\\"\\n\\n    if margin_percentage_points <= 0:\\n        raise ValueError(\\\"margin_percentage_points must be positive\\\")\\n    bootstrap = paired_bootstrap_accuracy_difference(\\n        base_records,\\n        adapter_records,\\n        iterations=iterations,\\n        seed=seed,\\n        stratify_by_subject=True,\\n    )\\n    lower = float(bootstrap[\\\"ci_lower_percentage_points\\\"])\\n    upper = float(bootstrap[\\\"ci_upper_percentage_points\\\"])\\n    threshold = -margin_percentage_points\\n    if lower > threshold:\\n        conclusion = \\\"no_material_forgetting\\\"\\n    elif upper < threshold:\\n        conclusion = \\\"forgetting_risk\\\"\\n    else:\\n        conclusion = \\\"inconclusive\\\"\\n    return {\\n        **bootstrap,\\n        \\\"noninferiority_margin_percentage_points\\\": margin_percentage_points,\\n        \\\"required_ci_lower_bound_above\\\": threshold,\\n        \\\"conclusion\\\": conclusion,\\n    }\\n\\n\\ndef representative_case_ids(\\n    base_records: Iterable[PredictionRecord],\\n    adapter_records: Iterable[PredictionRecord],\\n    *,\\n    limit: int = 10,\\n    seed: int = 3407,\\n) -> list[dict[str, str | None]]:\\n    \\\"\\\"\\\"Select content-safe IDs across improvement, regression, error, and parse categories.\\\"\\\"\\\"\\n\\n    if limit <= 0:\\n        raise ValueError(\\\"limit must be positive\\\")\\n    categories: dict[str, list[tuple[PredictionRecord, PredictionRecord]]] = defaultdict(list)\\n    for base, adapter in _aligned_pairs(base_records, adapter_records):\\n        if not base.parsed or not adapter.parsed:\\n            category = \\\"parse_failure\\\"\\n        elif not base.correct and adapter.correct:\\n            category = \\\"improved\\\"\\n        elif base.correct and not adapter.correct:\\n            category = \\\"regressed\\\"\\n        elif base.correct and adapter.correct:\\n            category = \\\"both_correct\\\"\\n        else:\\n            category = \\\"both_wrong\\\"\\n        categories[category].append((base, adapter))\\n\\n    order = [\\\"improved\\\", \\\"regressed\\\", \\\"both_wrong\\\", \\\"parse_failure\\\", \\\"both_correct\\\"]\\n    for category, pairs in categories.items():\\n        pairs.sort(\\n            key=lambda pair: hashlib.sha256(\\n                f\\\"{seed}:{category}:{pair[0].example_id}\\\".encode()\\n            ).hexdigest()\\n        )\\n\\n    selected: list[dict[str, str | None]] = []\\n    while len(selected) < limit:\\n        added = False\\n        for category in order:\\n            pairs = categories.get(category, [])\\n            if not pairs:\\n                continue\\n            base, adapter = pairs.pop(0)\\n            selected.append(\\n                {\\n                    \\\"example_id\\\": base.example_id,\\n                    \\\"subject\\\": base.subject or None,\\n                    \\\"gold\\\": base.gold,\\n                    \\\"base_prediction\\\": base.prediction,\\n                    \\\"adapter_prediction\\\": adapter.prediction,\\n                    \\\"category\\\": category,\\n                }\\n            )\\n            added = True\\n            if len(selected) == limit:\\n                break\\n        if not added:\\n            break\\n    return selected\\n\", \"tw_med_qlora/tmmlu.py\": \"\\\"\\\"\\\"Pinned TMMLU+ parsing and deterministic private evaluation materialization.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport csv\\nimport hashlib\\nimport json\\nfrom collections.abc import Iterable, Mapping, Sequence\\nfrom dataclasses import dataclass, replace\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nfrom tw_med_qlora.medqa import file_sha256\\nfrom tw_med_qlora.types import CHOICE_KEYS, MCQExample, stable_example_id\\n\\nTMMLU_FIELDS = frozenset({\\\"question\\\", \\\"A\\\", \\\"B\\\", \\\"C\\\", \\\"D\\\", \\\"answer\\\"})\\n\\n\\nclass TMMLUValidationError(ValueError):\\n    \\\"\\\"\\\"Raised when a TMMLU+ row or materialized split is malformed.\\\"\\\"\\\"\\n\\n\\n@dataclass(frozen=True)\\nclass SubjectExample:\\n    \\\"\\\"\\\"An MCQ with the subject kept out of the visible model prompt.\\\"\\\"\\\"\\n\\n    subject: str\\n    example: MCQExample\\n\\n    def __post_init__(self) -> None:\\n        if not self.subject:\\n            raise ValueError(\\\"subject must not be empty\\\")\\n\\n\\ndef _text(value: object, *, field: str) -> str:\\n    if not isinstance(value, str) or not value.strip():\\n        raise TMMLUValidationError(f\\\"{field} must be a non-empty string\\\")\\n    try:\\n        if value.encode(\\\"utf-8\\\").decode(\\\"utf-8\\\") != value:\\n            raise TMMLUValidationError(f\\\"{field} failed UTF-8 round trip\\\")\\n    except UnicodeError as error:\\n        raise TMMLUValidationError(f\\\"{field} must be UTF-8\\\") from error\\n    return value\\n\\n\\ndef tmmlu_row_to_example(\\n    row: Mapping[str, Any],\\n    *,\\n    subject: str,\\n    split: str,\\n    source: str,\\n    revision: str,\\n    row_index: int | None = None,\\n) -> SubjectExample:\\n    \\\"\\\"\\\"Convert a six-field source row without exposing subject metadata to the prompt.\\\"\\\"\\\"\\n\\n    if set(row) != TMMLU_FIELDS:\\n        raise TMMLUValidationError(\\n            f\\\"fields must be exactly {sorted(TMMLU_FIELDS)}; got {sorted(row)}\\\"\\n        )\\n    question = _text(row[\\\"question\\\"], field=\\\"question\\\")\\n    choices = {key: _text(row[key], field=key) for key in CHOICE_KEYS}\\n    answer = _text(row[\\\"answer\\\"], field=\\\"answer\\\").strip().upper()\\n    if answer not in CHOICE_KEYS:\\n        raise TMMLUValidationError(\\\"answer must be one of A, B, C, or D\\\")\\n    if row_index is not None and row_index < 0:\\n        raise TMMLUValidationError(\\\"row_index must be non-negative\\\")\\n    qualified_source = f\\\"{source}/{subject}\\\"\\n    id_source = (\\n        qualified_source if row_index is None else f\\\"{qualified_source}#source-row={row_index}\\\"\\n    )\\n    example = MCQExample(\\n        id=stable_example_id(\\n            source=id_source,\\n            revision=revision,\\n            split=split,\\n            question=question,\\n            choices=choices,\\n        ),\\n        source=qualified_source,\\n        revision=revision,\\n        split=split,\\n        question=question,\\n        choices=choices,\\n        answer=answer,\\n    )\\n    return SubjectExample(subject=subject, example=example)\\n\\n\\ndef read_tmmlu_csv(\\n    path: Path,\\n    *,\\n    subject: str,\\n    split: str,\\n    source: str,\\n    revision: str,\\n) -> list[SubjectExample]:\\n    \\\"\\\"\\\"Read one pinned subject CSV and validate every row.\\\"\\\"\\\"\\n\\n    examples: list[SubjectExample] = []\\n    with path.open(\\\"r\\\", encoding=\\\"utf-8-sig\\\", newline=\\\"\\\") as source_file:\\n        reader = csv.DictReader(source_file)\\n        if set(reader.fieldnames or []) != TMMLU_FIELDS:\\n            raise TMMLUValidationError(f\\\"unexpected CSV header: {reader.fieldnames}\\\")\\n        for row_index, row in enumerate(reader):\\n            examples.append(\\n                tmmlu_row_to_example(\\n                    row,\\n                    subject=subject,\\n                    split=split,\\n                    source=source,\\n                    revision=revision,\\n                    row_index=row_index,\\n                )\\n            )\\n    if not examples:\\n        raise TMMLUValidationError(f\\\"empty TMMLU+ file: {path.name}\\\")\\n    if len({item.example.id for item in examples}) != len(examples):\\n        raise AssertionError(f\\\"source-row stable IDs collided: {path.name}\\\")\\n    return examples\\n\\n\\ndef deterministic_order(\\n    examples: Iterable[SubjectExample],\\n    *,\\n    seed: int,\\n    purpose: str,\\n) -> list[SubjectExample]:\\n    \\\"\\\"\\\"Order examples by a version-independent SHA-256 key.\\\"\\\"\\\"\\n\\n    values = list(examples)\\n    return sorted(\\n        values,\\n        key=lambda item: hashlib.sha256(\\n            f\\\"{purpose}:{seed}:{item.example.id}\\\".encode()\\n        ).hexdigest(),\\n    )\\n\\n\\ndef stratified_calibration_sample(\\n    examples_by_subject: Mapping[str, Sequence[SubjectExample]],\\n    *,\\n    total: int,\\n    seed: int,\\n) -> dict[str, list[SubjectExample]]:\\n    \\\"\\\"\\\"Round-robin a deterministic sample so every configured subject is represented.\\\"\\\"\\\"\\n\\n    if total <= 0:\\n        raise ValueError(\\\"total must be positive\\\")\\n    subjects = list(examples_by_subject)\\n    if not subjects:\\n        raise ValueError(\\\"at least one subject is required\\\")\\n    if total < len(subjects):\\n        raise ValueError(\\\"calibration total must be at least the number of subjects\\\")\\n    ordered = {\\n        subject: deterministic_order(\\n            examples_by_subject[subject], seed=seed, purpose=\\\"calibration-sample\\\"\\n        )\\n        for subject in subjects\\n    }\\n    if any(not rows for rows in ordered.values()):\\n        raise ValueError(\\\"every subject must contain at least one example\\\")\\n\\n    selected = {subject: [] for subject in subjects}\\n    offsets = {subject: 0 for subject in subjects}\\n    while sum(len(rows) for rows in selected.values()) < total:\\n        added = False\\n        for subject in subjects:\\n            offset = offsets[subject]\\n            if offset >= len(ordered[subject]):\\n                continue\\n            selected[subject].append(ordered[subject][offset])\\n            offsets[subject] += 1\\n            added = True\\n            if sum(len(rows) for rows in selected.values()) == total:\\n                break\\n        if not added:\\n            raise ValueError(\\\"calibration total exceeds the available examples\\\")\\n    return selected\\n\\n\\ndef stability_sample(\\n    examples_by_subject: Mapping[str, Sequence[SubjectExample]],\\n    *,\\n    per_subject: int,\\n    sample_seed: int,\\n) -> dict[str, list[SubjectExample]]:\\n    \\\"\\\"\\\"Select the same at-most-N questions before applying each option seed.\\\"\\\"\\\"\\n\\n    if per_subject <= 0:\\n        raise ValueError(\\\"per_subject must be positive\\\")\\n    return {\\n        subject: deterministic_order(\\n            examples,\\n            seed=sample_seed,\\n            purpose=\\\"stability-sample\\\",\\n        )[:per_subject]\\n        for subject, examples in examples_by_subject.items()\\n    }\\n\\n\\ndef shuffle_options(item: SubjectExample, *, seed: int) -> SubjectExample:\\n    \\\"\\\"\\\"Apply a stable per-question choice permutation and remap the gold label.\\\"\\\"\\\"\\n\\n    example = item.example\\n    original_keys = sorted(\\n        CHOICE_KEYS,\\n        key=lambda key: hashlib.sha256(\\n            f\\\"option-order:{seed}:{example.id}:{key}\\\".encode()\\n        ).hexdigest(),\\n    )\\n    new_choices: dict[str, str] = {}\\n    new_answer = \\\"\\\"\\n    for new_key, old_key in zip(CHOICE_KEYS, original_keys, strict=True):\\n        new_choices[new_key] = example.choices[old_key]\\n        if old_key == example.answer:\\n            new_answer = new_key\\n    if not new_answer:\\n        raise AssertionError(\\\"option shuffle failed to remap the gold answer\\\")\\n    return replace(\\n        item,\\n        example=replace(example, choices=new_choices, answer=new_answer),\\n    )\\n\\n\\ndef _ids_sha256(examples: Sequence[SubjectExample]) -> str:\\n    digest = hashlib.sha256()\\n    for item in examples:\\n        digest.update(item.example.id.encode())\\n        digest.update(b\\\"\\\\n\\\")\\n    return digest.hexdigest()\\n\\n\\ndef write_twinkle_dataset(\\n    output_dir: Path,\\n    examples_by_subject: Mapping[str, Sequence[SubjectExample]],\\n    *,\\n    option_seed: int,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Write private Twinkle Eval JSONL plus a content-safe mapping manifest.\\\"\\\"\\\"\\n\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n    manifest: dict[str, Any] = {\\n        \\\"schema_version\\\": 1,\\n        \\\"option_seed\\\": option_seed,\\n        \\\"subjects\\\": {},\\n    }\\n    for subject, source_examples in examples_by_subject.items():\\n        shuffled = [shuffle_options(item, seed=option_seed) for item in source_examples]\\n        dataset_path = output_dir / f\\\"{subject}.jsonl\\\"\\n        temporary = dataset_path.with_suffix(\\\".jsonl.tmp\\\")\\n        with temporary.open(\\\"w\\\", encoding=\\\"utf-8\\\", newline=\\\"\\\\n\\\") as output_file:\\n            for item in shuffled:\\n                example = item.example\\n                row = {\\n                    \\\"question\\\": example.question,\\n                    **{key: example.choices[key] for key in CHOICE_KEYS},\\n                    \\\"answer\\\": example.answer,\\n                }\\n                output_file.write(json.dumps(row, ensure_ascii=False, separators=(\\\",\\\", \\\":\\\")))\\n                output_file.write(\\\"\\\\n\\\")\\n        temporary.replace(dataset_path)\\n        manifest[\\\"subjects\\\"][subject] = {\\n            \\\"count\\\": len(shuffled),\\n            \\\"ordered_ids_sha256\\\": _ids_sha256(shuffled),\\n            \\\"private_file_sha256\\\": file_sha256(dataset_path),\\n            \\\"id_by_line\\\": [item.example.id for item in shuffled],\\n        }\\n    manifest[\\\"total\\\"] = sum(\\n        int(subject[\\\"count\\\"]) for subject in manifest[\\\"subjects\\\"].values()\\n    )\\n    return manifest\\n\", \"tw_med_qlora/phase4.py\": \"\\\"\\\"\\\"Phase 4 workload, adapter, and Colab serving safety helpers.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport math\\nimport zipfile\\nfrom collections.abc import Sequence\\nfrom dataclasses import asdict, dataclass\\nfrom pathlib import Path, PurePosixPath\\nfrom typing import Any\\n\\n\\n@dataclass(frozen=True)\\nclass EvaluationWorkload:\\n    \\\"\\\"\\\"Predeclared request counts for the two evaluation tracks.\\\"\\\"\\\"\\n\\n    medqa_full: int\\n    tmmlu_full: int\\n    tmmlu_stability: int\\n\\n    @property\\n    def total(self) -> int:\\n        return self.medqa_full + self.tmmlu_full + self.tmmlu_stability\\n\\n    def as_dict(self) -> dict[str, int]:\\n        return {**asdict(self), \\\"total\\\": self.total}\\n\\n\\ndef phase4_workload(\\n    *,\\n    medqa_test_rows: int,\\n    tmmlu_test_rows: int,\\n    full_model_count: int,\\n    subject_count: int,\\n    stability_examples_per_subject: int,\\n    stability_seeds: Sequence[int],\\n    stability_model_count: int,\\n) -> EvaluationWorkload:\\n    \\\"\\\"\\\"Calculate the approved Phase 4 generation count without hidden multipliers.\\\"\\\"\\\"\\n\\n    values = {\\n        \\\"medqa_test_rows\\\": medqa_test_rows,\\n        \\\"tmmlu_test_rows\\\": tmmlu_test_rows,\\n        \\\"full_model_count\\\": full_model_count,\\n        \\\"subject_count\\\": subject_count,\\n        \\\"stability_examples_per_subject\\\": stability_examples_per_subject,\\n        \\\"stability_model_count\\\": stability_model_count,\\n    }\\n    if any(value <= 0 for value in values.values()):\\n        raise ValueError(f\\\"workload values must be positive: {values}\\\")\\n    if not stability_seeds:\\n        raise ValueError(\\\"at least one stability seed is required\\\")\\n    if len(set(stability_seeds)) != len(stability_seeds):\\n        raise ValueError(\\\"stability seeds must be unique\\\")\\n\\n    return EvaluationWorkload(\\n        medqa_full=medqa_test_rows * full_model_count,\\n        tmmlu_full=tmmlu_test_rows * full_model_count,\\n        tmmlu_stability=(\\n            subject_count\\n            * stability_examples_per_subject\\n            * len(stability_seeds)\\n            * stability_model_count\\n        ),\\n    )\\n\\n\\ndef project_evaluation_cost(\\n    *,\\n    workload: EvaluationWorkload,\\n    measured_requests: int,\\n    measured_inference_seconds: float,\\n    measured_server_startup_seconds: float,\\n    planned_server_starts: int,\\n    compute_units_per_hour: float,\\n    buffer_fraction: float = 0.20,\\n    price_per_compute_unit: float | None = None,\\n    currency: str | None = None,\\n) -> dict[str, int | float | str | None]:\\n    \\\"\\\"\\\"Project full evaluation from measured generation and server-start timings.\\\"\\\"\\\"\\n\\n    numeric = (\\n        measured_inference_seconds,\\n        measured_server_startup_seconds,\\n        compute_units_per_hour,\\n        buffer_fraction,\\n    )\\n    if measured_requests <= 0 or planned_server_starts <= 0:\\n        raise ValueError(\\\"request and server-start counts must be positive\\\")\\n    if any(not math.isfinite(value) for value in numeric):\\n        raise ValueError(\\\"cost inputs must be finite\\\")\\n    if measured_inference_seconds <= 0 or measured_server_startup_seconds < 0:\\n        raise ValueError(\\\"measured timings are invalid\\\")\\n    if compute_units_per_hour <= 0 or buffer_fraction < 0:\\n        raise ValueError(\\\"compute-unit rate must be positive and buffer non-negative\\\")\\n    if price_per_compute_unit is not None and price_per_compute_unit < 0:\\n        raise ValueError(\\\"price per compute unit cannot be negative\\\")\\n\\n    seconds_per_request = measured_inference_seconds / measured_requests\\n    projected_seconds = (\\n        seconds_per_request * workload.total\\n        + measured_server_startup_seconds * planned_server_starts\\n    )\\n    projected_hours = projected_seconds / 3600\\n    projected_compute_units = projected_hours * compute_units_per_hour\\n    buffered_compute_units = projected_compute_units * (1 + buffer_fraction)\\n    estimated_cost = (\\n        buffered_compute_units * price_per_compute_unit\\n        if price_per_compute_unit is not None\\n        else None\\n    )\\n    return {\\n        \\\"measured_requests\\\": measured_requests,\\n        \\\"measured_seconds_per_request\\\": seconds_per_request,\\n        \\\"planned_server_starts\\\": planned_server_starts,\\n        \\\"projected_seconds\\\": projected_seconds,\\n        \\\"projected_hours\\\": projected_hours,\\n        \\\"compute_units_per_hour_user_input\\\": compute_units_per_hour,\\n        \\\"projected_compute_units\\\": projected_compute_units,\\n        \\\"buffer_fraction\\\": buffer_fraction,\\n        \\\"projected_compute_units_with_buffer\\\": buffered_compute_units,\\n        \\\"price_per_compute_unit_user_input\\\": price_per_compute_unit,\\n        \\\"estimated_cost\\\": estimated_cost,\\n        \\\"currency_user_input\\\": currency,\\n    }\\n\\n\\ndef _file_sha256(path: Path) -> str:\\n    digest = hashlib.sha256()\\n    with path.open(\\\"rb\\\") as source:\\n        for block in iter(lambda: source.read(1024 * 1024), b\\\"\\\"):\\n            digest.update(block)\\n    return digest.hexdigest()\\n\\n\\ndef extract_verified_adapter(\\n    archive_path: Path,\\n    destination: Path,\\n    *,\\n    expected_sha256: str,\\n    expected_bytes: int,\\n    expected_base_model_id: str,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Verify and safely extract the reviewed Phase 3 adapter archive.\\\"\\\"\\\"\\n\\n    if not archive_path.is_file():\\n        raise FileNotFoundError(f\\\"adapter archive not found: {archive_path}\\\")\\n    actual_bytes = archive_path.stat().st_size\\n    if actual_bytes != expected_bytes:\\n        raise ValueError(\\n            f\\\"adapter archive size mismatch: expected={expected_bytes}, actual={actual_bytes}\\\"\\n        )\\n    actual_sha256 = _file_sha256(archive_path)\\n    if actual_sha256 != expected_sha256:\\n        raise ValueError(\\n            \\\"adapter archive SHA-256 mismatch: \\\"\\n            f\\\"expected={expected_sha256}, actual={actual_sha256}\\\"\\n        )\\n\\n    destination.mkdir(parents=True, exist_ok=True)\\n    resolved_destination = destination.resolve()\\n    with zipfile.ZipFile(archive_path) as archive:\\n        for member in archive.infolist():\\n            member_path = PurePosixPath(member.filename.replace(\\\"\\\\\\\\\\\", \\\"/\\\"))\\n            if member_path.is_absolute() or \\\"..\\\" in member_path.parts:\\n                raise ValueError(f\\\"unsafe adapter archive member: {member.filename}\\\")\\n            target = (destination / Path(*member_path.parts)).resolve()\\n            if target != resolved_destination and resolved_destination not in target.parents:\\n                raise ValueError(f\\\"archive member escapes destination: {member.filename}\\\")\\n        archive.extractall(destination)\\n\\n    adapter_configs = list(destination.rglob(\\\"adapter_config.json\\\"))\\n    if len(adapter_configs) != 1:\\n        raise ValueError(\\n            \\\"expected exactly one adapter_config.json; \\\"\\n            f\\\"found={len(adapter_configs)}\\\"\\n        )\\n    config_path = adapter_configs[0]\\n    config = json.loads(config_path.read_text(encoding=\\\"utf-8\\\"))\\n    actual_base = config.get(\\\"base_model_name_or_path\\\")\\n    if actual_base != expected_base_model_id:\\n        raise ValueError(\\n            \\\"adapter/base mismatch: \\\"\\n            f\\\"expected={expected_base_model_id}, actual={actual_base}\\\"\\n        )\\n    weight_files = sorted(config_path.parent.glob(\\\"adapter_model*.safetensors\\\"))\\n    if not weight_files:\\n        raise ValueError(\\\"adapter archive does not contain safetensors weights\\\")\\n    return {\\n        \\\"archive_sha256\\\": actual_sha256,\\n        \\\"archive_bytes\\\": actual_bytes,\\n        \\\"adapter_dir\\\": str(config_path.parent),\\n        \\\"base_model_id\\\": actual_base,\\n        \\\"adapter_config_sha256\\\": _file_sha256(config_path),\\n        \\\"weight_files\\\": [path.name for path in weight_files],\\n    }\\n\\n\\ndef build_vllm_serve_command(\\n    *,\\n    model_id: str,\\n    model_revision: str,\\n    served_model_name: str,\\n    port: int,\\n    max_model_length: int,\\n    gpu_memory_utilization: float,\\n    seed: int,\\n    adapter_name: str | None = None,\\n    adapter_path: Path | None = None,\\n    max_lora_rank: int = 16,\\n) -> list[str]:\\n    \\\"\\\"\\\"Build the pinned Colab-only vLLM CLI without embedding secrets.\\\"\\\"\\\"\\n\\n    if not model_id or not model_revision or not served_model_name:\\n        raise ValueError(\\\"model identifiers must not be empty\\\")\\n    if not 1 <= port <= 65535:\\n        raise ValueError(\\\"port must be between 1 and 65535\\\")\\n    if max_model_length <= 0 or max_lora_rank <= 0:\\n        raise ValueError(\\\"model length and LoRA rank must be positive\\\")\\n    if not 0 < gpu_memory_utilization < 1:\\n        raise ValueError(\\\"gpu memory utilization must be between zero and one\\\")\\n    if (adapter_name is None) != (adapter_path is None):\\n        raise ValueError(\\\"adapter name and path must be provided together\\\")\\n\\n    command = [\\n        \\\"vllm\\\",\\n        \\\"serve\\\",\\n        model_id,\\n        \\\"--revision\\\",\\n        model_revision,\\n        \\\"--tokenizer-revision\\\",\\n        model_revision,\\n        \\\"--served-model-name\\\",\\n        served_model_name,\\n        \\\"--port\\\",\\n        str(port),\\n        \\\"--dtype\\\",\\n        \\\"bfloat16\\\",\\n        \\\"--quantization\\\",\\n        \\\"bitsandbytes\\\",\\n        \\\"--max-model-len\\\",\\n        str(max_model_length),\\n        \\\"--gpu-memory-utilization\\\",\\n        str(gpu_memory_utilization),\\n        \\\"--generation-config\\\",\\n        \\\"vllm\\\",\\n        \\\"--language-model-only\\\",\\n        \\\"--seed\\\",\\n        str(seed),\\n    ]\\n    if adapter_path is not None and adapter_name is not None:\\n        command.extend(\\n            [\\n                \\\"--enable-lora\\\",\\n                \\\"--max-lora-rank\\\",\\n                str(max_lora_rank),\\n                \\\"--lora-modules\\\",\\n                json.dumps(\\n                    {\\\"name\\\": adapter_name, \\\"path\\\": str(adapter_path)},\\n                    separators=(\\\",\\\", \\\":\\\"),\\n                ),\\n            ]\\n        )\\n    return command\\n\"}")
HELPER_ROOT = Path("/content/tw-med-eval-helpers")
for relative_name, source in EMBEDDED_HELPER_FILES.items():
    target = HELPER_ROOT / relative_name
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
if str(HELPER_ROOT) not in sys.path:
    sys.path.insert(0, str(HELPER_ROOT))

from tw_med_qlora.evaluation import (
    PredictionRecord,
    accuracy_summary,
    parse_mcq_answer,
)
from tw_med_qlora.phase4 import (
    build_vllm_serve_command,
    extract_verified_adapter,
    phase4_workload,
    project_evaluation_cost,
)
from tw_med_qlora.tmmlu import (
    read_tmmlu_csv,
    shuffle_options,
    stratified_calibration_sample,
    write_twinkle_dataset,
)
print("Repository-tested Phase 4 helpers loaded.")


In [ ]:
evaluation_config = PROJECT_CONFIG["evaluation"]
workload_config = evaluation_config["workload"]
all_subjects = (
    evaluation_config["medical_subjects"]
    + evaluation_config["control_subjects"]
)
WORKLOAD = phase4_workload(
    medqa_test_rows=int(workload_config["medqa_test_rows"]),
    tmmlu_test_rows=int(workload_config["tmmlu_test_rows"]),
    full_model_count=int(workload_config["full_model_count"]),
    subject_count=len(all_subjects),
    stability_examples_per_subject=int(
        evaluation_config["stability_examples_per_subject"]
    ),
    stability_seeds=evaluation_config["stability_seeds"],
    stability_model_count=int(workload_config["stability_model_count"]),
)
if WORKLOAD.total != int(workload_config["expected_total_requests"]):
    raise RuntimeError("Phase 4 request-count contract changed unexpectedly")
print(json.dumps(WORKLOAD.as_dict(), ensure_ascii=False, indent=2))


## 4. 驗證並解開 Phase 3 adapter

這裡只接受已審核的 Drive ZIP、固定大小與 SHA-256，並檢查
`adapter_config.json` 的 base model ID。


In [ ]:
adapter_contract = evaluation_config["phase3_adapter"]
adapter_archive = Path(adapter_contract["drive_archive"])
adapter_audit = extract_verified_adapter(
    adapter_archive,
    Path("/content/tw-med-phase4-adapter"),
    expected_sha256=adapter_contract["archive_sha256"],
    expected_bytes=int(adapter_contract["archive_bytes"]),
    expected_base_model_id=adapter_contract["base_model_id"],
)
ADAPTER_DIR = Path(adapter_audit["adapter_dir"])
print(json.dumps(adapter_audit, ensure_ascii=False, indent=2))


## 5. 只下載 TMMLU+ validation，固定抽 20 題

所有 13 科至少各一題。此 cell 的 allow-pattern 只允許 `_val.csv`；若目錄出現
`_test.csv` 會立刻停止。


In [ ]:
tmmlu_config = PROJECT_CONFIG["data"]["tmmluplus"]
tmmlu_root = Path("/content/tmmluplus-validation")
snapshot_path = Path(
    snapshot_download(
        repo_id=tmmlu_config["dataset_id"],
        repo_type="dataset",
        revision=tmmlu_config["revision"],
        allow_patterns=["data/*_val.csv"],
        local_dir=tmmlu_root,
        token=HF_TOKEN,
    )
)
forbidden_test_files = list(snapshot_path.rglob("*_test.csv"))
if forbidden_test_files:
    raise RuntimeError(
        f"Calibration downloaded forbidden test files: {forbidden_test_files}"
    )

validation_by_subject = {}
validation_counts = {}
for subject in all_subjects:
    path = snapshot_path / "data" / f"{subject}_val.csv"
    rows = read_tmmlu_csv(
        path,
        subject=subject,
        split="validation",
        source=tmmlu_config["dataset_id"],
        revision=tmmlu_config["revision"],
    )
    validation_by_subject[subject] = rows
    validation_counts[subject] = len(rows)

calibration_by_subject = stratified_calibration_sample(
    validation_by_subject,
    total=int(evaluation_config["calibration_examples"]),
    seed=int(PROJECT_CONFIG["project"]["seed"]),
)
calibration_manifest = write_twinkle_dataset(
    PRIVATE_ROOT / "tmmlu-calibration",
    calibration_by_subject,
    option_seed=int(evaluation_config["full_shuffle_seed"]),
)
if calibration_manifest["total"] != 20:
    raise RuntimeError(
        "Calibration must contain exactly 20 unique validation questions"
    )
calibration_ids = [
    item.example.id
    for subject in all_subjects
    for item in calibration_by_subject[subject]
]
if len(set(calibration_ids)) != 20:
    raise RuntimeError("Calibration sample contains duplicate IDs")
print(json.dumps({
    "split": "validation",
    "subjects": len(all_subjects),
    "rows": calibration_manifest["total"],
    "ordered_ids_sha256": hashlib.sha256(
        "\n".join(calibration_ids).encode("utf-8")
    ).hexdigest(),
    "validation_counts": validation_counts,
    "test_files_loaded": 0,
}, ensure_ascii=False, indent=2))


## 6. Twinkle Eval extractor/scorer 與 vLLM server 工具


In [ ]:
from openai import OpenAI
from twinkle_eval.metrics.extractors.box import BoxExtractor
from twinkle_eval.metrics.scorers.exact import ExactMatchScorer

box_extractor = BoxExtractor()
exact_scorer = ExactMatchScorer()
if box_extractor.extract(r"\boxed{A}") != "A":
    raise RuntimeError("Twinkle Eval box extractor contract failed")
if not exact_scorer.score("A", "A") or exact_scorer.score("A", "B"):
    raise RuntimeError("Twinkle Eval exact scorer contract failed")

ACTIVE_SERVER = None
ACTIVE_LOG_HANDLE = None

def wait_for_server(process, *, port, timeout_seconds=1800):
    started = time.perf_counter()
    last_notice = -30
    while time.perf_counter() - started < timeout_seconds:
        if process.poll() is not None:
            raise RuntimeError(
                f"vLLM server exited with code {process.returncode}; "
                "inspect its private log"
            )
        try:
            with urllib.request.urlopen(
                f"http://127.0.0.1:{port}/health", timeout=5
            ) as response:
                if response.status == 200:
                    return time.perf_counter() - started
        except (urllib.error.URLError, TimeoutError):
            pass
        elapsed = int(time.perf_counter() - started)
        if elapsed - last_notice >= 30:
            print(f"Waiting for vLLM server: {elapsed}s elapsed")
            last_notice = elapsed
        time.sleep(5)
    raise TimeoutError(f"vLLM server did not become healthy within {timeout_seconds}s")

def start_server(command, *, label, port):
    global ACTIVE_SERVER, ACTIVE_LOG_HANDLE
    if ACTIVE_SERVER is not None:
        raise RuntimeError("A vLLM server is already active")
    log_path = LOG_ROOT / f"{label}.log"
    ACTIVE_LOG_HANDLE = log_path.open("w", encoding="utf-8")
    environment = os.environ.copy()
    environment["HF_TOKEN"] = HF_TOKEN
    ACTIVE_SERVER = subprocess.Popen(
        command,
        stdout=ACTIVE_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        env=environment,
        text=True,
        start_new_session=True,
    )
    try:
        startup_seconds = wait_for_server(ACTIVE_SERVER, port=port)
    except Exception:
        ACTIVE_LOG_HANDLE.flush()
        tail = log_path.read_text(encoding="utf-8", errors="replace")[-6000:]
        print(tail)
        stop_server()
        raise
    print(f"{label} server ready after {startup_seconds:.1f}s")
    return startup_seconds

def stop_server():
    global ACTIVE_SERVER, ACTIVE_LOG_HANDLE
    process = ACTIVE_SERVER
    ACTIVE_SERVER = None
    stopped_running_process = False
    if process is not None and process.poll() is None:
        stopped_running_process = True
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=30)
        except subprocess.TimeoutExpired:
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait(timeout=10)
    log_handle = ACTIVE_LOG_HANDLE
    ACTIVE_LOG_HANDLE = None
    if log_handle is not None and not log_handle.closed:
        log_handle.close()
    if stopped_running_process:
        time.sleep(5)

def visible_prompt(item):
    example = item.example
    choices = "\n".join(
        f"{key}. {example.choices[key]}" for key in ("A", "B", "C", "D")
    )
    return f"{example.question}\n\n{choices}"

generation_config = evaluation_config["generation"]
GENERATION_MAX_TOKENS = int(generation_config["max_tokens"])
MINIMUM_PARSE_RATE = float(
    generation_config["minimum_calibration_parse_rate"]
)
TOKEN_LIMIT_HITS_FAIL_CALIBRATION = bool(
    generation_config["token_limit_hits_fail_calibration"]
)
if generation_config["token_limit_hits_count_as_incorrect"] is not True:
    raise RuntimeError("Token-limit outputs must remain strict parse failures")
SYSTEM_PROMPT = (
    "請選擇唯一最佳答案。不要解釋或重述題目；只輸出單一大寫 "
    r"A–D 字母，或一個 LaTeX 答案框，例如 \boxed{A}。"
)

def evaluate_one(client, *, model_name, public_label, item):
    started = time.perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": visible_prompt(item)},
        ],
        temperature=float(generation_config["temperature"]),
        top_p=float(generation_config["top_p"]),
        max_tokens=GENERATION_MAX_TOKENS,
        seed=int(PROJECT_CONFIG["project"]["seed"]),
    )
    latency = time.perf_counter() - started
    raw_output = response.choices[0].message.content or ""
    prediction = parse_mcq_answer(raw_output)
    twinkle_box_prediction = box_extractor.extract(raw_output)
    if (
        prediction is not None
        and r"\boxed" in raw_output
        and twinkle_box_prediction != prediction
    ):
        raise RuntimeError(
            "Strict parser and Twinkle BoxExtractor disagree on a boxed response"
        )
    if prediction is not None:
        exact_match = bool(
            exact_scorer.score(prediction, item.example.answer)
        )
        if exact_match != (prediction == item.example.answer):
            raise RuntimeError("Twinkle exact-match scorer contract drifted")
    usage = response.usage
    public = PredictionRecord(
        example_id=item.example.id,
        model=public_label,
        source=item.example.source,
        subject=item.subject,
        gold=item.example.answer,
        prediction=prediction,
        raw_output_sha256=hashlib.sha256(raw_output.encode("utf-8")).hexdigest(),
        latency_seconds=latency,
        prompt_tokens=getattr(usage, "prompt_tokens", None),
        completion_tokens=getattr(usage, "completion_tokens", None),
    )
    private = {
        "example_id": item.example.id,
        "model": public_label,
        "subject": item.subject,
        "gold": item.example.answer,
        "raw_output": raw_output,
    }
    return public, private

calibration_items = [
    shuffle_options(
        item,
        seed=int(evaluation_config["full_shuffle_seed"]),
    )
    for subject in all_subjects
    for item in calibration_by_subject[subject]
]

def evaluate_model(*, port, served_name, public_label):
    client = OpenAI(api_key="local-eval", base_url=f"http://127.0.0.1:{port}/v1")
    started = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        futures = [
            executor.submit(
                evaluate_one,
                client,
                model_name=served_name,
                public_label=public_label,
                item=item,
            )
            for item in calibration_items
        ]
        pairs = [future.result() for future in futures]
    elapsed = time.perf_counter() - started
    public_records = sorted((pair[0] for pair in pairs), key=lambda row: row.example_id)
    private_records = sorted(
        (pair[1] for pair in pairs), key=lambda row: row["example_id"]
    )
    private_path = PRIVATE_ROOT / f"{public_label}-raw.jsonl"
    with private_path.open("w", encoding="utf-8", newline="\n") as target:
        for record in private_records:
            target.write(json.dumps(record, ensure_ascii=False, separators=(",", ":")))
            target.write("\n")
    return public_records, elapsed

print("Twinkle Eval and vLLM calibration helpers are ready.")


## 7. 校準三個模型

會啟動兩次 server：原廠模型一次；台灣 base 與 adapter 共用另一次。
模型首次下載可能需要數分鐘，cell 每 30 秒會印一次等待狀態。


In [ ]:
model_config = PROJECT_CONFIG["models"]["primary"]
vllm_config = evaluation_config["vllm"]
PORT = 8000
startup_timings = []
inference_timings = {}
public_records_by_model = {}

original_name = "original-instruct"
original_command = build_vllm_serve_command(
    model_id=model_config["baseline_id"],
    model_revision=model_config["baseline_revision"],
    served_model_name=original_name,
    port=PORT,
    max_model_length=int(vllm_config["max_model_length"]),
    gpu_memory_utilization=float(vllm_config["gpu_memory_utilization"]),
    seed=int(PROJECT_CONFIG["project"]["seed"]),
)
try:
    startup_timings.append(
        start_server(original_command, label=original_name, port=PORT)
    )
    records, elapsed = evaluate_model(
        port=PORT,
        served_name=original_name,
        public_label=original_name,
    )
    public_records_by_model[original_name] = records
    inference_timings[original_name] = elapsed
finally:
    stop_server()

localized_name = "localized-base"
adapter_name = "localized-medical-adapter"
localized_command = build_vllm_serve_command(
    model_id=model_config["model_id"],
    model_revision=model_config["revision"],
    served_model_name=localized_name,
    port=PORT,
    max_model_length=int(vllm_config["max_model_length"]),
    gpu_memory_utilization=float(vllm_config["gpu_memory_utilization"]),
    seed=int(PROJECT_CONFIG["project"]["seed"]),
    adapter_name=adapter_name,
    adapter_path=ADAPTER_DIR,
    max_lora_rank=int(vllm_config["max_lora_rank"]),
)
try:
    startup_timings.append(
        start_server(localized_command, label="localized-with-adapter", port=PORT)
    )
    for served_name, public_label in (
        (localized_name, localized_name),
        (adapter_name, adapter_name),
    ):
        records, elapsed = evaluate_model(
            port=PORT,
            served_name=served_name,
            public_label=public_label,
        )
        public_records_by_model[public_label] = records
        inference_timings[public_label] = elapsed
finally:
    stop_server()

if set(public_records_by_model) != {original_name, localized_name, adapter_name}:
    raise RuntimeError("All three calibration models must complete")
print(json.dumps({
    "startup_seconds": startup_timings,
    "inference_seconds": inference_timings,
    "requests": sum(len(rows) for rows in public_records_by_model.values()),
}, ensure_ascii=False, indent=2))


## 8. 安全摘要、成本預估與 Drive 封存


In [ ]:
calibration_summary = {
    "label": "validation calibration only; not a Phase 4 test result",
    "split": "validation",
    "unique_questions": 20,
    "models": {
        model_name: {
            **accuracy_summary(records),
            "completion_tokens_total": sum(
                record.completion_tokens or 0 for record in records
            ),
            "max_token_limit_hits": sum(
                record.completion_tokens is not None
                and record.completion_tokens >= GENERATION_MAX_TOKENS
                for record in records
            ),
        }
        for model_name, records in public_records_by_model.items()
    },
    "generation_contract": {
        "parser": "standalone_A-D_or_exactly_one_simple_boxed_A-D",
        "scorer": exact_scorer.get_name(),
        "max_tokens": GENERATION_MAX_TOKENS,
        "minimum_parse_rate": MINIMUM_PARSE_RATE,
        "token_limit_hits_fail_calibration": (
            TOKEN_LIMIT_HITS_FAIL_CALIBRATION
        ),
        "token_limit_hits_count_as_incorrect": True,
    },
    "inference_seconds": inference_timings,
    "server_startup_seconds": startup_timings,
}
parse_gate_failures = {
    model_name: summary["parse_rate"]
    for model_name, summary in calibration_summary["models"].items()
    if summary["parse_rate"] < MINIMUM_PARSE_RATE
}
observed_token_limit_hits = {
    model_name: summary["max_token_limit_hits"]
    for model_name, summary in calibration_summary["models"].items()
    if summary["max_token_limit_hits"] > 0
}
generation_gate_passed = not (
    parse_gate_failures
    or (
        TOKEN_LIMIT_HITS_FAIL_CALIBRATION
        and observed_token_limit_hits
    )
)
calibration_summary["generation_gate"] = {
    "passed": generation_gate_passed,
    "parse_rate_failures": parse_gate_failures,
    "observed_max_token_limit_hits": observed_token_limit_hits,
    "max_token_limit_failures": (
        observed_token_limit_hits
        if TOKEN_LIMIT_HITS_FAIL_CALIBRATION
        else {}
    ),
    "failure_action": (
        None
        if generation_gate_passed
        else "Do not unlock full evaluation; review archived evidence."
    ),
}
measured_requests = sum(len(rows) for rows in public_records_by_model.values())
cost_estimate = project_evaluation_cost(
    workload=WORKLOAD,
    measured_requests=measured_requests,
    measured_inference_seconds=sum(inference_timings.values()),
    measured_server_startup_seconds=sum(startup_timings) / len(startup_timings),
    planned_server_starts=2,
    compute_units_per_hour=float(COMPUTE_UNITS_PER_HOUR),
    price_per_compute_unit=PRICE_PER_COMPUTE_UNIT,
    currency=CURRENCY_LABEL,
)
public_predictions = {
    model_name: [record.as_public_dict() for record in records]
    for model_name, records in public_records_by_model.items()
}
(PUBLIC_ROOT / "calibration_summary.json").write_text(
    json.dumps(
        calibration_summary, ensure_ascii=False, indent=2, allow_nan=False
    )
    + "\n",
    encoding="utf-8",
)
(PUBLIC_ROOT / "public_predictions.json").write_text(
    json.dumps(
        public_predictions, ensure_ascii=False, indent=2, allow_nan=False
    )
    + "\n",
    encoding="utf-8",
)
(PUBLIC_ROOT / "cost_estimate.json").write_text(
    json.dumps(cost_estimate, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
with (PRIVATE_ROOT / "pip-freeze.txt").open("w", encoding="utf-8") as target:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        check=True,
        stdout=target,
    )

private_archive_base = RUN_ROOT / f"{RUN_ID}-phase4-calibration-private"
private_archive = Path(
    shutil.make_archive(
        str(private_archive_base),
        "zip",
        root_dir=RUN_ROOT,
        base_dir="private",
    )
)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as source:
        for block in iter(lambda: source.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

manifest = {
    "schema_version": 1,
    "phase": 4,
    "run_mode": RUN_MODE,
    "created_at_utc": datetime.now(UTC).isoformat(),
    "full_evaluation_unlocked": False,
    "test_files_loaded": 0,
    "project_seed": PROJECT_CONFIG["project"]["seed"],
    "hardware": hardware_audit,
    "models": {
        "original": {
            "id": model_config["baseline_id"],
            "revision": model_config["baseline_revision"],
        },
        "localized_base": {
            "id": model_config["model_id"],
            "revision": model_config["revision"],
        },
        "adapter": adapter_audit,
    },
    "data": {
        "dataset_id": tmmlu_config["dataset_id"],
        "revision": tmmlu_config["revision"],
        "split": "validation",
        "unique_questions": 20,
        "subject_count": len(all_subjects),
        "ordered_ids_sha256": hashlib.sha256(
            "\n".join(calibration_ids).encode("utf-8")
        ).hexdigest(),
    },
    "dependencies": dependency_audit,
    "twinkle_eval_contract": {
        "repository": evaluation_config["twinkle_eval"]["repository"],
        "revision": evaluation_config["twinkle_eval"]["revision"],
        "extractor": (
            "strict standalone A-D or exactly one simple boxed A-D; "
            f"Twinkle audit={box_extractor.get_name()}"
        ),
        "scorer": exact_scorer.get_name(),
        "shuffle_options_inside_runner": False,
    },
    "workload": WORKLOAD.as_dict(),
    "calibration_summary": calibration_summary,
    "cost_estimate": cost_estimate,
    "private_archive": {
        "sha256": file_sha256(private_archive),
        "bytes": private_archive.stat().st_size,
    },
}
manifest_path = PUBLIC_ROOT / "run_manifest.json"
manifest_path.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)

drive_root = Path(
    "/content/drive/MyDrive/tw-med-llm-qlora/phase4/calibration/runs"
)
drive_root.mkdir(parents=True, exist_ok=True)
drive_archive = drive_root / private_archive.name
drive_manifest = drive_root / f"{RUN_ID}-run-manifest.json"
drive_summary = drive_root / f"{RUN_ID}-calibration-summary.json"
shutil.copy2(private_archive, drive_archive)
shutil.copy2(manifest_path, drive_manifest)
shutil.copy2(PUBLIC_ROOT / "calibration_summary.json", drive_summary)
if file_sha256(drive_archive) != manifest["private_archive"]["sha256"]:
    raise RuntimeError("Drive private archive SHA-256 verification failed")

receipt = {
    "phase": 4,
    "run_mode": "calibration",
    "drive_private_archive": str(drive_archive),
    "drive_manifest": str(drive_manifest),
    "drive_calibration_summary": str(drive_summary),
    "archive_sha256": manifest["private_archive"]["sha256"],
    "archive_bytes": manifest["private_archive"]["bytes"],
    "full_evaluation_unlocked": False,
}
receipt_path = PUBLIC_ROOT / "receipt.json"
receipt_path.write_text(
    json.dumps(receipt, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
    encoding="utf-8",
)
drive_receipt = drive_root / f"{RUN_ID}-receipt.json"
shutil.copy2(receipt_path, drive_receipt)

print(json.dumps(calibration_summary, ensure_ascii=False, indent=2))
print(json.dumps(cost_estimate, ensure_ascii=False, indent=2))
print(
    json.dumps(
        {**receipt, "drive_receipt": str(drive_receipt)},
        ensure_ascii=False,
        indent=2,
    )
)
if generation_gate_passed:
    print("\nPhase 4 的 20 題 validation 校準完成；生成協定閘門通過。")
else:
    print(
        "\nPhase 4 校準檔案已安全封存，但生成協定閘門未通過；"
        "完整評估仍鎖定。"
    )
print(
    "請下載 run_manifest.json、receipt.json、calibration_summary.json，"
    "以及 phase4-calibration-private.zip 回傳確認；私人 ZIP 會留在 "
    "gitignored 目錄，不會提交 Git。"
)
